# 02. Updating figures/ and tables/

Runs after `01_demandas_revisores.ipynb` (sections 05 to 17 already executed, with the dated CSVs and figures written to `analysis/resultados/`). This notebook recomputes nothing: it reads what is already in `resultados/` and does two things.

1. Archives whatever exists in `jhpn_revisions/figures/` and `jhpn_revisions/tables/` into a timestamped zip inside the `_old/` folder of each. Nothing is deleted without a backup, and the backup stays inside `_old/` (never a loose `_bkp_` next to the current file).
2. Writes the current version: the 6 final figures (PNG at 200 dpi + PDF) to `figures/`, and Tables 1, 2 and 3 formatted as `.docx` (not only CSV) to `tables/`.

Figure 3 (PROBAST+AI, 17 Aug 2026) is drawn by hand and does not exist in `resultados/`; the archiving step leaves its current file in `figures/` untouched.

## 01. Mount and paths

In [ ]:
import os
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/USP/Doutorado/Coorte de pelotas/01_artigos/artigo_1'
    AMBIENTE = 'colab'
except ModuleNotFoundError:
    BASE = '../..'
    AMBIENTE = 'local'

OUT_DIR = f'{BASE}/jhpn_revisions/analysis/resultados'
FIGURAS = f'{BASE}/jhpn_revisions/Final_review/figures'
TABELAS = f'{BASE}/jhpn_revisions/Final_review/tables'
DATA = '20260826'          # round of the CSVs and of figures 1, 2, 2b, S1 and S2
DATA_FIG4 = '20260831'     # Figure 4 redone on 31 Aug (two panels, predictor presence)
os.makedirs(FIGURAS, exist_ok=True)
os.makedirs(TABELAS, exist_ok=True)
print(AMBIENTE, '| BASE =', BASE)

In [ ]:
%pip install -q python-docx
import shutil, zipfile, datetime
import pandas as pd
from docx import Document
from docx.shared import Pt, Cm
from docx.enum.section import WD_ORIENT

## 02. Archive what is already there

Zips everything currently at the top level of `figures/` and of `tables/` (without descending into `_old/`) into a single `.zip` per folder, saved as `<folder>/_old/<folder>_bkp_<timestamp>.zip`. The files that the next section will overwrite are removed after the zip is written; what will not be regenerated (Figure 3) is preserved and kept out of the removal list.

In [ ]:
def arquivar_pasta(pasta, preservar=()):
    nome_pasta = os.path.basename(os.path.normpath(pasta))
    old_dir = f'{pasta}/_old'
    os.makedirs(old_dir, exist_ok=True)
    arquivos = [f for f in os.listdir(pasta)
                if os.path.isfile(f'{pasta}/{f}') and not f.startswith('.')]
    if not arquivos:
        print(f'{nome_pasta}/: nothing to archive')
        return
    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    zip_path = f'{old_dir}/{nome_pasta}_bkp_{ts}.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in arquivos:
            z.write(f'{pasta}/{f}', arcname=f)
    print(f'{nome_pasta}/_old/{os.path.basename(zip_path)}  ({len(arquivos)} files)')
    removidos = [f for f in arquivos if f not in preservar]
    for f in removidos:
        os.remove(f'{pasta}/{f}')
    mantidos = set(arquivos) - set(removidos)
    if mantidos:
        print(f'  preserved in {nome_pasta}/ (not regenerated by this notebook): {sorted(mantidos)}')

FIG3 = {'fig3_probastai_20260817.png', 'fig3_probastai_20260817.pdf'}
arquivar_pasta(FIGURAS, preservar=FIG3)
arquivar_pasta(TABELAS)

## 03. Figure 4 redone (31 Aug 2026)

The Figure 4 of 26 Aug counted the field `3 Most Important Features`, reported in only 26 of the 39 combinations, and measured the *importance* declared by the study. The new version counts *presence* of the predictor in the list of candidates (`Predictor Types` in v7), available for all 38 studies, with the study as the unit, and comes out in two panels:

- **(A)** the 22 studies in which anthropometry of the same dimension as the outcome appears among the predictors, the group that the submitted version of the manuscript called *data leakage* and that the revised text now calls diagnostic tautology. The group comes from the `antropometria_preditores` column of `leakage_recontado.csv`, produced by section 09 of notebook 01.
- **(B)** the 37 studies that contribute an eligible outcome, that is, the overall prevalence of the predictor.

The canonicalisation below resolves the spelling variants used by the articles (`age in months`, `child age`, `usia`) into a single name, and discards the family labels of the extraction (`child`, `household`, `demographic`), which are groupers and not predictors. Coverage: 330 of the 436 items that are predictors proper (76%); what remains is a tail of single items, with no effect on the first ten. The counts go to `fig4_preditores_painelA.csv` and `fig4_preditores_painelB.csv`.

In [ ]:
# -- 03.1 new Figure 4: most present candidate predictors, in two panels --
# Replaces the version of 26 Aug, which counted only the field "3 Most Important Features"
# (top-3 reported in 26 of the 39 combinations). The new version counts PRESENCE of the predictor
# in the list of candidates ("Predictor Types" in v7), available for all 38 studies, and the
# unit becomes the STUDY, not the study x outcome combination.
#   Panel A: the 22 studies in which anthropometry of the same dimension as the outcome appears
#             among the predictors (the group the submitted manuscript called leakage).
#   Panel B: the 37 studies that contribute an eligible outcome.
import re
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

V7 = f'{BASE}/methods/extracao_artigo1_v7.xlsx'
VERM, AZUL, TINTA = '#C2453A', '#2A6FB0', '#222222'

# family labels from the extraction (these are not predictors)
STOP = re.compile(
    r'^(child|maternal|paternal|parental|household|family|demographic|sociodemographic|'
    r'socio-?economic|socioeconomic|geographic|anthropometric|child anthropometric|'
    r'child demographic|child sociodemographic|child characteristics|child morbidity|'
    r'clinical|behavioural|behavioral|dietary|environmental|nutritional|wash|household/wash|'
    r'child anthropometric/demographic|household/socioeconomic|others?|other variables?|'
    r'variables?|predictors?|nr|na|not reported|none|z-score|categories|etc)$')

CANONICO = [
    (r'^(child\s+)?(age at measurement|age in months|age \(months\)|age)$|child.*\bage\b|\bage in months\b', 'Child age'),
    (r'\bsex\b|gender', 'Child sex'),
    (r'weight[-/ ]for[-/ ](height|length)|\bwhz\b|\bw\s*/\s*h\b', 'Weight-for-height'),
    (r'weight[-/ ]for[-/ ]age|\bwaz\b|\bw\s*/\s*a\b|^weight/age$', 'Weight-for-age'),
    (r'height[-/ ]for[-/ ]age|\bhaz\b|\bh\s*/\s*a\b|tb\s*/\s*u', 'Height-for-age'),
    (r'birth\s*_?\s*(weight|wt)|weight at birth', 'Birth weight'),
    (r'birth\s*_?\s*(length|height|size)|size at birth', 'Birth length or size'),
    (r'(mid[- ]?)?upper[- ]?arm circumference|\bmuac\b|\blila\b|lingkar lengan', 'MUAC'),
    (r'head circumference', 'Head circumference'),
    (r'\bbmi\b|body mass|\bimc\b|\bzbmi\b|\bbaz\b', 'Body mass index'),
    (r'^(child\s+)?(height|length|stature)\b|height in cm|child.*(height|length)', 'Height or length'),
    (r'^(child\s+)?weight\b|child.*weight|weight in kg', 'Weight'),
    (r"(maternal|mother'?s?)\s*(bmi|body mass)", 'Maternal BMI'),
    (r"(maternal|mother'?s?)\s*(height|stature)", 'Maternal height'),
    (r"(maternal|mother'?s?)\s*educ|mother.*schooling", 'Maternal education'),
    (r"(paternal|father'?s?|partner)\s*educ", 'Paternal education'),
    (r"(maternal|mother'?s?)\s*age|age of mother", 'Maternal age'),
    (r"(maternal|mother'?s?|paternal|father'?s?)\s*(occupation|employ|work)", 'Parental occupation'),
    (r'^education$|educational (level|attainment)|schooling|couple education|'
     r'education of mother|^couple$|^partner$', 'Parental education'),
    (r'antenatal|prenatal (care|visit)', 'Antenatal care'),
    (r'place of delivery|delivery place|delivery mode|mode of delivery|caesar|cesar|'
     r"place of (child'?s? )?birth", 'Delivery'),
    (r'birth order|birth rank|parity', 'Birth order or parity'),
    (r'birth interval|preceding interval', 'Birth interval'),
    (r'twin|multiple birth', 'Twin status'),
    (r'gestational age|preterm|premature', 'Gestational age'),
    (r'breast\s*-?\s*feed|breastfeeding|exclusive breast', 'Breastfeeding'),
    (r'complementary feeding|weaning|dietary diversity|food (intake|consumption|frequency)|'
     r'diet\b|meal|nutrition intake', 'Diet and feeding'),
    (r'fast[- ]?food', 'Fast-food intake'),
    (r'vitamin|supplement|micronutrient|iron|zinc|deworm', 'Micronutrient supplementation'),
    (r'immuni[sz]|vaccin', 'Immunisation'),
    (r'wealth|asset index|\bincome\b|poverty|economic status', 'Household wealth or income'),
    (r'household size|family size|number of (children|household|siblings)|sibling', 'Household size'),
    (r'toilet|sanitation|latrine', 'Sanitation'),
    (r'(drinking )?water(\s+(source|supply))?$|water source', 'Water source'),
    (r'\bregion\b|province|district|state\b|geographic', 'Region'),
    (r'residen|urban|rural', 'Place of residence'),
    (r'television|\btv\b|radio|refrigerator|motorcycle|electricity|asset ownership', 'Household assets'),
    (r'floor|roof|wall|housing|dwelling|room', 'Housing conditions'),
    (r'ethnic|race|caste|religion', 'Ethnicity or religion'),
    (r'health insurance|insurance', 'Health insurance'),
    (r'marital|married', 'Marital status'),
    (r'^(occupation|employment|work status)$', 'Occupation'),
    (r'diarrh', 'Diarrhoea'),
    (r'fever\b', 'Fever'),
    (r'cough|ari\b|respiratory infection', 'Respiratory symptoms'),
    (r'anaemi|anemi|h(a)?emoglobin', 'Anaemia'),
    (r'physical activity|exercise|sedentary|screen time|sleep', 'Physical activity or sleep'),
    (r'smok|tobacco|alcohol', 'Smoking or alcohol'),
]
ANTRO_DESF = {'Height or length', 'Weight', 'Body mass index', 'Height-for-age',
              'Weight-for-age', 'Weight-for-height'}


def itens_pred(txt):
    t = str(txt)
    if t.strip().lower() in ('nan', 'nr', ''):
        return []
    t = re.sub(r'\(([^)]*)\)', r', \1', t)   # what sits inside parentheses is an item too
    return [p.strip(' .') for p in re.split(r'[;,]| and (?=[a-z])', t) if len(p.strip(' .')) > 1]


def canoniza_pred(txt):
    t = re.sub(r'\s+', ' ', str(txt).strip().lower().strip(' .:-'))
    if not t or STOP.match(t):
        return None
    for padrao, nome in CANONICO:
        if re.search(padrao, t):
            return nome
    return None


In [ ]:
# -- 03.2 counting and plotting --
lk = pd.read_csv(f'{OUT_DIR}/leakage_classification.csv', sep=';', encoding='utf-8-sig')
ids_todos = sorted(lk['ID'].unique())
ids_antro = sorted(lk[lk['anthropometry_among_predictors']]['ID'].unique())

v7 = pd.read_excel(V7, sheet_name='Per Combination', header=1)
# AA_10 (Kar, 2021) modelled only a composite undernutrition outcome, with no isolable
# anthropometric dimension. It is ineligible, and leaves every count.
EXCLUIDOS = ['AA_10']
v7 = v7[~v7['ID'].isin(EXCLUIDOS)].copy()
v7 = v7[v7['ID'].notna()].drop_duplicates(subset=['ID']).set_index('ID')


def conta_presenca(ids):
    linhas = []
    for i in ids:
        nomes = {canoniza_pred(x) for x in itens_pred(v7.loc[i, 'Predictor Types'])}
        nomes.discard(None)
        linhas += [{'ID': i, 'predictor': n} for n in nomes]
    c = (pd.DataFrame(linhas).groupby('predictor')['ID'].nunique().rename('studies')
         .sort_values(ascending=False).reset_index())
    c['pct'] = (100 * c['studies'] / len(ids)).round(0).astype(int)
    c['outcome_dimension_anthropometry'] = c['predictor'].isin(ANTRO_DESF)
    return c


A, B = conta_presenca(ids_antro), conta_presenca(ids_todos)
A.to_csv(f'{OUT_DIR}/figure4_predictors_panel_a.csv', sep=';', index=False, encoding='utf-8-sig')
B.to_csv(f'{OUT_DIR}/figure4_predictors_panel_b.csv', sep=';', index=False, encoding='utf-8-sig')

plt.rcParams.update({'font.family': 'DejaVu Sans', 'text.color': TINTA,
                     'axes.labelcolor': TINTA, 'xtick.color': TINTA, 'ytick.color': TINTA,
                     'axes.edgecolor': '#666666', 'savefig.dpi': 200})
fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.9))
for ax, cont, n, letra, tit in [
        (axes[0], A, len(ids_antro), 'A',
         'Studies with outcome-dimension anthropometry\namong the predictors'),
        (axes[1], B, len(ids_todos), 'B',
         'All studies contributing an eligible outcome\n(overall predictor prevalence)')]:
    top = cont.head(10).iloc[::-1]
    barras = ax.barh(top['predictor'], top['studies'],
                     color=[VERM if a else AZUL for a in top['outcome_dimension_anthropometry']], height=0.68)
    for b, e, p in zip(barras, top['studies'], top['pct']):
        ax.text(b.get_width() + n * 0.015, b.get_y() + b.get_height() / 2,
                f'{e} ({p}%)', va='center', fontsize=8.5, color=TINTA)
    ax.set_xlim(0, n * 1.20)
    ax.set_xlabel(f'Studies reporting the predictor (n = {n})', fontsize=9)
    ax.set_title(f'({letra}) {tit}', fontsize=9.5, fontweight='bold', loc='left', pad=8)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True, nbins=5))
    ax.tick_params(labelsize=9)
    ax.set_axisbelow(True)
    ax.xaxis.grid(True, color='#DDDDDD')
    for lado in ('top', 'right'):
        ax.spines[lado].set_visible(False)
mp = [plt.Rectangle((0, 0), 1, 1, color=VERM), plt.Rectangle((0, 0), 1, 1, color=AZUL)]
fig.legend(mp, ['Anthropometry sharing the outcome dimension', 'Other predictors'],
           loc='lower center', ncol=2, frameon=False, fontsize=9,
           handlelength=1.1, handleheight=1.1, bbox_to_anchor=(0.5, -0.045))
fig.tight_layout()
for ext in ('png', 'pdf'):
    fig.savefig(f'{OUT_DIR}/fig4_preditores_{DATA_FIG4}.{ext}', bbox_inches='tight')
plt.show()
print(f'resultados/fig4_preditores_{DATA_FIG4}.png/.pdf | A: {len(ids_antro)} studies | '
      f'B: {len(ids_todos)} studies')
print(A.head(10).to_string(index=False))


## 04. Final figures (copied from `resultados/` with the current date stamp)

Same logic as section 16 of `01_demandas_revisores.ipynb`. Figure 4 comes from section 03 above, with its own date stamp (`DATA_FIG4`).

In [ ]:
# fig2b (the standalone map) entered on 30 Aug; Figure 4 has its own date because it was redone later
FINAIS = {'fig1_prisma': DATA, 'fig2_distribuicao': DATA, 'fig2b_mapa': DATA,
          'fig4_preditores': DATA_FIG4, 'figS1_auc_desfecho': DATA, 'figS2_desempenho_preditor': DATA}
for nome, data_fig in FINAIS.items():
    for ext in ('png', 'pdf'):
        arq = f'{OUT_DIR}/{nome}_{data_fig}.{ext}'
        if os.path.exists(arq):
            shutil.copy2(arq, f'{FIGURAS}/{nome}_{data_fig}.{ext}')
            print(f'figures/{nome}_{data_fig}.{ext}')
        else:
            print(f'MISSING from resultados/: {nome}_{data_fig}.{ext}')
print('kept unchanged: fig3_probastai_20260817.png/.pdf')

## 05. Formatted tables (.docx) in `tables/`

Table 1 (studies), Table 2 (performance by outcome) and Table 3 (certainty of the evidence, GRADE), from the CSVs that section 17 of notebook 01 already writes to `resultados/`. Calibri font, `Table Grid` style, bold header; Table 1 comes out in landscape because it has 11 columns.

In [ ]:
def novo(paisagem=False):
    d = Document()
    st = d.styles['Normal']; st.font.name = 'Calibri'; st.font.size = Pt(10)
    s = d.sections[0]
    s.left_margin = s.right_margin = Cm(1.5)
    if paisagem:
        s.orientation = WD_ORIENT.LANDSCAPE
        s.page_width, s.page_height = s.page_height, s.page_width
    return d

def titulo(d, txt):
    p = d.add_paragraph(); r = p.add_run(txt)
    r.font.size = Pt(12); r.bold = True
    p.paragraph_format.space_after = Pt(8)

def tabela(d, header, rows, fonte=8):
    t = d.add_table(rows=1, cols=len(header)); t.style = 'Table Grid'
    for c, h in zip(t.rows[0].cells, header):
        c.text = ''
        r = c.paragraphs[0].add_run(str(h)); r.bold = True; r.font.size = Pt(fonte)
    for row in rows:
        cells_ = t.add_row().cells
        for c, v in zip(cells_, row):
            c.text = ''
            r = c.paragraphs[0].add_run('' if pd.isna(v) else str(v)); r.font.size = Pt(fonte)
    return t

def ler(nome):
    return pd.read_csv(f'{OUT_DIR}/{nome}', sep=';', encoding='utf-8-sig')

In [ ]:
t1 = ler('table1_study_characteristics.csv')
d = novo(paisagem=True)
titulo(d, 'Table 1. Characteristics and performance of the 39 eligible study-outcome combinations')
tabela(d, t1.columns.tolist(), t1.values.tolist(), fonte=7)
d.save(f'{TABELAS}/table1_study_characteristics.docx')
print('tables/table1_study_characteristics.docx', t1.shape)

In [ ]:
t2 = ler('table2_performance_by_outcome.csv')
d = novo()
titulo(d, 'Table 2. Model performance and heterogeneity by nutritional outcome')
tabela(d, t2.columns.tolist(), t2.values.tolist(), fonte=8)
d.save(f'{TABELAS}/table2_performance_by_outcome.docx')
print('tables/table2_performance_by_outcome.docx', t2.shape)

In [ ]:
t3 = ler('table3_grade_certainty.csv')
d = novo()
titulo(d, 'Table 3. Certainty of the evidence by nutritional outcome (adapted GRADE)')
tabela(d, t3.columns.tolist(), t3.values.tolist(), fonte=9)
d.save(f'{TABELAS}/table3_grade_certainty.docx')
print('tables/table3_grade_certainty.docx', t3.shape)

## 06. Next step

1. Check `figures/` and `tables/` (open the `.docx` files to check line breaks and column widths).
2. If anything is wrong, the previous files are in the zip inside `figures/_old/` and `tables/_old/`; there is no need to rerun notebook 01 just for that.
3. Rerun whenever notebook 01 produces a new round of `resultados/` (change the `DATA` constant above to the date of the new round before running).